# CrashLens + PyTorch Integration

This notebook demonstrates tracking PyTorch training jobs with CrashLens.

## What you'll learn:
- ✅ Track PyTorch training loops
- ✅ Handle GPU OOM errors
- ✅ Get AI-powered fix recommendations
- ✅ Visualize GPU metrics during training

## 1. Setup

In [ ]:
# Install dependencies
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

from crashlens.jupyter import JupyterWorkloadTracker

# Initialize tracker
tracker = JupyterWorkloadTracker("https://invigorating-empathy-production-dee5.up.railway.app")

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Define a Simple Neural Network

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

print("✓ Model defined")

## 3. Create Synthetic Dataset

In [ ]:
# Create synthetic data
input_size = 784  # 28x28 images
hidden_size = 256
num_classes = 10
num_samples = 1000
batch_size = 32

# Random data
X = torch.randn(num_samples, input_size)
y = torch.randint(0, num_classes, (num_samples,))

# Create dataset and dataloader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Dataset size: {len(dataset)} samples")
print(f"Batch size: {batch_size}")
print(f"Number of batches: {len(dataloader)}")

## 4. Successful Training (Tracked)

In [ ]:
with tracker.track("PyTorch Training - 5 Epochs", display_metrics=True):
    # Initialize model
    model = SimpleNN(input_size, hidden_size, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print("Starting training...\n")
    
    # Training loop
    num_epochs = 5
    for epoch in range(num_epochs):
        epoch_loss = 0
        
        for batch_idx, (data, targets) in enumerate(dataloader):
            data = data.to(device)
            targets = targets.to(device)
            
            # Forward pass
            outputs = model(data)
            loss = criterion(outputs, targets)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    
    print("\n✓ Training completed successfully!")

## 5. Simulate GPU OOM Error

Let's intentionally create a GPU OOM scenario and see how CrashLens diagnoses it:

In [ ]:
if torch.cuda.is_available():
    try:
        with tracker.track("GPU OOM Test", auto_diagnose=True):
            print("Attempting to allocate huge tensor on GPU...")
            # This will likely fail with OOM on most GPUs
            huge_tensor = torch.randn(50000, 50000).cuda()
            
    except RuntimeError as e:
        print(f"\n⚠️  RuntimeError caught: {str(e)[:100]}...")
        print("Check the AI diagnosis above for fix recommendations!")
else:
    print("⚠️  No GPU available. Skipping OOM test.")
    print("This test requires a CUDA-enabled GPU.")

## 6. Simulate Dependency Error

In [ ]:
try:
    with tracker.track("Missing Package Test", auto_diagnose=True):
        print("Trying to import torchvision.models...")
        # This might fail if torchvision is not installed
        from torchvision import models
        print("✓ torchvision available")
        
except ImportError as e:
    print(f"\n⚠️  ImportError: {e}")
    print("Check the diagnosis for installation instructions!")

## 7. Best Practices for PyTorch + CrashLens

### Track Different Training Stages

You can track data loading, training, and validation separately:

In [ ]:
# Track data loading
with tracker.track("Data Loading"):
    print("Loading dataset...")
    # Your data loading code
    print("✓ Data loaded")

# Track model initialization
with tracker.track("Model Initialization"):
    print("Initializing model...")
    model = SimpleNN(input_size, hidden_size, num_classes)
    print("✓ Model initialized")

# Track training
with tracker.track("Training Loop"):
    print("Training model...")
    # Your training code
    print("✓ Training complete")

### Memory Management Tips

When working with GPUs, track memory-intensive operations:

In [ ]:
if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"GPU Memory cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    
    # Clear cache
    torch.cuda.empty_cache()
    print("\n✓ GPU cache cleared")
else:
    print("No GPU available")

## 8. Common PyTorch Failures CrashLens Can Diagnose

1. **GPU Out of Memory**
   - Batch size too large
   - Model too big for GPU
   - Memory leak in training loop

2. **CUDA Errors**
   - Device mismatch (CPU vs GPU tensors)
   - Driver version issues
   - Invalid CUDA operations

3. **Dependency Errors**
   - Missing torch/torchvision
   - Version incompatibilities
   - Missing CUDA toolkit

4. **Data Loading Issues**
   - Missing dataset files
   - Corrupted data
   - Incorrect file paths

5. **Model Loading Errors**
   - Missing checkpoint files
   - Incompatible model architectures
   - State dict mismatches

## Next Steps

- Try training on real datasets (MNIST, CIFAR-10)
- Experiment with larger models (ResNet, BERT)
- Track distributed training across multiple GPUs
- View all workloads: https://frontend-zeta-eight-92.vercel.app/dashboard